In [1]:
import os

In [2]:
%pwd

'd:\\PredictBot-Score-MLOps\\research'

In [3]:
os.chdir('..')

In [4]:
%pwd

'd:\\PredictBot-Score-MLOps'

In [24]:
from src.predictor_bot_score.config.configuration import yaml_load , create_directories
from src.predictor_bot_score.logger import logger
from src.predictor_bot_score.constants import CONFIG_PATH
from pathlib import Path
from datetime import datetime
from dataclasses import dataclass
import pandas as pd 
import os

In [6]:
@dataclass(frozen=True)
class DataTransformationConfig:
    validated_data_path: Path
    transformed_data_dir: Path

In [19]:
class Config_manager:

    def __init__(self , config = CONFIG_PATH):

        self.config_path = yaml_load(config)

        create_directories([self.config_path.artifacts_root])
    
    def get_data_transformation_config(self):

        config = self.config_path.data_transformation

        create_directories([config.transformed_data])

        data_transformation_config = DataTransformationConfig(
            validated_data_path=Path(config.validated_data_path),
            transformed_data_dir =Path(config.transformed_data)
        )
        
        return data_transformation_config




In [ ]:
class Data_transformation:

    def __init__(self, config: DataTransformationConfig):
        self.config = config
        self.data = self.read_data()
    def read_data(self):
        try:
            logger.info(f"Attempting to read data from: {self.config.validated_data_path}")
            data = pd.read_csv(self.config.validated_data_path)
            logger.info("Data successfully loaded into memory.")
            return data
        
        except FileNotFoundError:
            # This is your custom message
            print("--- ALERT: The file is missing! Please check the path. ---")
            
            # This is the log file entry
            logger.error(f"File not found at: {self.config.validated_data_path}")
            
             

    def transformed_data(self):
        try:
            logger.info("=" * 50)
            logger.info("DATA TRANSFORMATION PIPELINE STARTED")
            logger.info("=" * 50)

            self.data["timestamp"] = pd.to_datetime(self.data["timestamp"], utc=True)

            self.data["bot_score"] = self.data["bot_score"].astype(float)

            timestamp = datetime.now().strftime("%Y_%m_%d_%H_%M")
            out_path  = self.config.transformed_data_dir / f"_{timestamp}.csv"


            self.data.to_csv(out_path,index=False)
            logger.info(f"Data saved  {out_path}")
            logger.info(f"DATA TRANSFORMATION DONE")

        
        except Exception as e:
            logger.info(e)

In [42]:
transformation_config = Config_manager().get_data_transformation_config()
transformation_config = Data_transformation(transformation_config)
transformation_config.transformed_data()

[2026-06-10 21:08:24,506: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-06-10 21:08:24,520: INFO: common: Directory created (or already exists) at: artifacts]
[2026-06-10 21:08:24,522: INFO: common: Directory created (or already exists) at: artifacts/data_transformation/transformed_data]
[2026-06-10 21:08:24,546: INFO: 3623165930: Attempting to read data from: artifacts\data_validation\validated\validated_2026_06_10_18_31.csv]
[2026-06-10 21:08:24,685: INFO: 3623165930: Data successfully loaded into memory.]
[2026-06-10 21:08:24,689: INFO: 3623165930: ==================================================]
[2026-06-10 21:08:24,689: INFO: 3623165930: DATA TRANSFORMATION PIPELINE STARTED]
[2026-06-10 21:08:24,689: INFO: 3623165930: ==================================================]
[2026-06-10 21:08:25,118: INFO: 3623165930: Data saved  artifacts\data_transformation\transformed_data\_2026_06_10_21_08.csv]
